<a href="https://colab.research.google.com/github/Kaveesha-Vihanga/DS_Project/blob/component-3/Final_implementation_ZEROSHOT_ADVISOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

file_path = '/content/drive/MyDrive/DSGP Component 3/reviews_with_aspect_sentiments_ZEROSHOT.csv'

df = pd.read_csv(file_path)
print("Data loaded. Shape:", df.shape)
print("Columns:", df.columns.tolist()[:10])  # show first 10 columns

Data loaded. Shape: (16252, 31)
Columns: ['User Name', 'Country', 'Room Info', 'Stay Date', 'Traveller Type', 'Rating', 'Review Date', 'Address', 'review_full', 'review_clean']


In [3]:
# Find all columns that end with ' positive' or ' negative'
aspect_cols = [col for col in df.columns if col.endswith(' positive') or col.endswith(' negative')]

# Extract aspect names (remove the ' positive' or ' negative' suffix)
aspects = sorted(set([col.rsplit(' ', 1)[0] for col in aspect_cols]))
print(f"Found {len(aspects)} aspects: {aspects}")

# If no aspect columns found, you might need to check your column names.
if len(aspects) == 0:
    raise ValueError("No aspect‑sentiment columns found. Make sure your CSV includes columns like 'rooms positive' etc.")

Found 9 aspects: ['amenities', 'check-in/out', 'cleanliness', 'facilities', 'food', 'location', 'rooms', 'staff', 'value for money']


In [4]:
hotel_col = 'Address'

# Check if the column exists
if hotel_col not in df.columns:
    print(f"Warning: '{hotel_col}' not found. Available columns:", df.columns.tolist())
    raise ValueError("Please set hotel_col to the correct hotel identifier column.")

# Group by hotel
hotel_groups = df.groupby(hotel_col)

hotel_stats = []

for hotel, group in hotel_groups:
    total_reviews = len(group)
    avg_rating = group['Rating'].mean()   # assuming 'Rating' column exists
    record = {
        'hotel': hotel,
        'total_reviews': total_reviews,
        'avg_rating': avg_rating
    }

    for aspect in aspects:
        pos_col = f"{aspect} positive"
        neg_col = f"{aspect} negative"
        if pos_col in group.columns and neg_col in group.columns:
            pos_count = group[pos_col].sum()
            neg_count = group[neg_col].sum()
            record[f"{aspect}_pos_rate"] = pos_count / total_reviews
            record[f"{aspect}_neg_rate"] = neg_count / total_reviews
            record[f"{aspect}_net"] = (pos_count - neg_count) / total_reviews

    hotel_stats.append(record)

hotel_df = pd.DataFrame(hotel_stats)
print(f"Aggregated data for {len(hotel_df)} hotels.")
hotel_df.head()

Aggregated data for 76 hotels.


,hotel,total_reviews,avg_rating,amenities_pos_rate,amenities_neg_rate,amenities_net,check-in/out_pos_rate,check-in/out_neg_rate,check-in/out_net,cleanliness_pos_rate,...,location_net,rooms_pos_rate,rooms_neg_rate,rooms_net,staff_pos_rate,staff_neg_rate,staff_net,value for money_pos_rate,value for money_neg_rate,value for money_net
0,"50 Albert Pl, Dehiwala-Mount Lavinia 10350",35,8.657143,0.771429,0.171429,0.600000,0.771429,0.200000,0.571429,0.657143,...,0.600000,0.800000,0.228571,0.571429,0.742857,0.171429,0.571429,0.771429,0.200000,0.571429
1,519@jamtree,13,9.307692,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.923077,...,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000
2,"55 Lionel Edirisinghe Mawatha, Colombo 00500",78,9.166667,0.935897,0.064103,0.871795,0.961538,0.166667,0.794872,0.641026,...,0.884615,0.935897,0.064103,0.871795,0.935897,0.012821,0.923077,0.961538,0.025641,0.935897
3,AHAS GAWWA,44,8.886364,0.954545,0.113636,0.840909,0.954545,0.113636,0.840909,0.613636,...,0.818182,0.954545,0.159091,0.795455,0.909091,0.090909,0.818182,0.954545,0.068182,0.886364
4,Aathma Colombo House,159,9.729560,0.981132,0.050314,0.930818,0.974843,0.050314,0.924528,0.647799,...,0.924528,0.987421,0.044025,0.943396,0.949686,0.012579,0.937107,0.962264,0.031447,0.930818


In [5]:
# Create feature names
pos_features = [f"{aspect}_pos_rate" for aspect in aspects if f"{aspect}_pos_rate" in hotel_df.columns]
neg_features = [f"{aspect}_neg_rate" for aspect in aspects if f"{aspect}_neg_rate" in hotel_df.columns]
all_features = pos_features + neg_features

# Prepare data
X = hotel_df[all_features]
y = hotel_df['avg_rating']

# Add constant
X = sm.add_constant(X)

# Fit OLS
model_asym = sm.OLS(y, X).fit()
print(model_asym.summary())

# Extract coefficients for later use
pos_coef = {feat: model_asym.params[feat] for feat in pos_features}
neg_coef = {feat: model_asym.params[feat] for feat in neg_features}

                            OLS Regression Results                            
Dep. Variable:             avg_rating   R-squared:                       0.936
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     46.68
Date:                Fri, 13 Mar 2026   Prob (F-statistic):           1.58e-27
Time:                        08:41:36   Log-Likelihood:                 10.887
No. Observations:                  76   AIC:                             16.23
Df Residuals:                      57   BIC:                             60.51
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [19]:
def rating_to_rates(rating, scale=10):
    """Convert a rating (1–scale) to expected positive/negative mention rates."""
    rating = max(1, min(scale, rating))
    pos_rate = 0.1 + 0.8 * ((rating - 1) / (scale - 1))
    neg_rate = 0.9 - 0.8 * ((rating - 1) / (scale - 1))
    return pos_rate, neg_rate

def diagnose_hotel(hotel_name, hotel_df, model, pos_coef, neg_coef, aspects):
    """Print investment recommendations (ratings on 10‑point scale)."""
    hotel_row = hotel_df[hotel_df['hotel'] == hotel_name].iloc[0]

    print(f"\n== Diagnosis for {hotel_name} ==")
    print(f"Current average rating: {hotel_row['avg_rating']:.2f} / 10")

    # Prepare features for prediction, ensuring correct order and presence of 'const'
    predict_row_data = pd.Series(index=model.params.index, dtype=float)
    predict_row_data['const'] = 1.0

    for col in model.params.index[1:]:
        predict_row_data[col] = hotel_row.get(col, 0.0) # Get value or default to 0 if feature is unexpectedly missing

    X_pred = pd.DataFrame([predict_row_data])
    pred = model.predict(X_pred)[0]
    print(f"Predicted rating: {pred:.2f} / 10")

    # Compute impacts
    impacts = []
    for aspect in aspects:
        pos_rate = hotel_row.get(f"{aspect}_pos_rate", 0)
        neg_rate = hotel_row.get(f"{aspect}_neg_rate", 0)
        pos_impact = pos_coef.get(f"{aspect}_pos_rate", 0) * pos_rate
        neg_impact = neg_coef.get(f"{aspect}_neg_rate", 0) * neg_rate
        net_impact = pos_impact + neg_impact
        impacts.append({
            'aspect': aspect,
            'pos_rate': pos_rate,
            'neg_rate': neg_rate,
            'pos_impact': pos_impact,
            'neg_impact': neg_impact,
            'net_impact': net_impact
        })

    impacts_df = pd.DataFrame(impacts).sort_values('net_impact')

    print("\n**Strengths (positive impact):**")
    strengths = impacts_df[impacts_df['net_impact'] > 0].sort_values('net_impact', ascending=False).head(3)
    for _, row in strengths.iterrows():
        print(f"  {row['aspect']}: +{row['net_impact']:.3f} (pos rate {row['pos_rate']:.1%}, neg rate {row['neg_rate']:.1%})")

    print("\n**Weaknesses (negative impact):**")
    weaknesses = impacts_df[impacts_df['net_impact'] < 0].sort_values('net_impact').head(3)
    for _, row in weaknesses.iterrows():
        print(f"  {row['aspect']}: {row['net_impact']:.3f} (pos rate {row['pos_rate']:.1%}, neg rate {row['neg_rate']:.1%})")

    print("\n**Top investment priorities:**")
    priorities = []
    for aspect in aspects:
        neg_rate = hotel_row.get(f"{aspect}_neg_rate", 0)
        neg_coef_val = neg_coef.get(f"{aspect}_neg_rate", 0)
        pos_coef_val = pos_coef.get(f"{aspect}_pos_rate", 0)

        gain_from_reducing_neg = -neg_coef_val * 0.1   # per 10% reduction
        gain_from_increasing_pos = pos_coef_val * 0.1

        priorities.append({
            'aspect': aspect,
            'gain_reduce_neg': gain_from_reducing_neg,
            'gain_increase_pos': gain_from_increasing_pos,
            'neg_rate': neg_rate,
            'pos_rate': hotel_row.get(f"{aspect}_pos_rate", 0)
        })

    priorities_df = pd.DataFrame(priorities).sort_values('gain_reduce_neg', ascending=False)
    for _, row in priorities_df.head(3).iterrows():
        print(f"  {row['aspect']}: reduce negative mentions (gain ~{row['gain_reduce_neg']:.3f} per 10% reduction) - current neg rate {row['neg_rate']:.1%}")

    return impacts_df, priorities_df

def score_new_hotel(user_ratings, model_asym, pos_coef, neg_coef, aspects):
    """Predict rating (on 10‑point scale) for a new hotel based on user's 1‑10 aspect ratings."""
    # Build feature vector from user's 1‑10 ratings
    features = {}
    for aspect in aspects:
        rating = user_ratings.get(aspect, 5)
        pos_rate, neg_rate = rating_to_rates(rating, scale=10)
        features[f"{aspect}_pos_rate"] = pos_rate
        features[f"{aspect}_neg_rate"] = neg_rate

    # Prepare prediction row with the correct order (including constant)
    predict_row = pd.Series(index=model_asym.params.index, dtype=float)
    predict_row['const'] = 1.0
    for col in model_asym.params.index[1:]:          # all feature columns
        predict_row[col] = features.get(col, 0.0)    # fill from features dict

    X_new = pd.DataFrame([predict_row])
    pred = model_asym.predict(X_new)[0]

    print(f"\n**Predicted overall rating for your new hotel: {pred:.2f} / 10**")
    print(f" (Equivalent to {pred/10*100:.1f}% satisfaction score)\n")

    print("**To increase this score, consider investing in:**")
    suggestions = []
    for aspect in aspects:
        current_rating = user_ratings.get(aspect, 5)
        # Only consider improvement if not already at maximum
        if current_rating < 10:
            new_rating = current_rating + 1
            pos_new, neg_new = rating_to_rates(new_rating, scale=10)
            pos_old, neg_old = rating_to_rates(current_rating, scale=10)
            delta_pos = (pos_new - pos_old) * pos_coef.get(f"{aspect}_pos_rate", 0)
            delta_neg = (neg_new - neg_old) * neg_coef.get(f"{aspect}_neg_rate", 0)
            total_gain = delta_pos + delta_neg
        else:
            total_gain = 0.0   # already perfect, no gain possible

        suggestions.append({
            'aspect': aspect,
            'current_rating': current_rating,
            'gain_per_1pt_increase': total_gain,
        })

    suggestions_df = pd.DataFrame(suggestions).sort_values('gain_per_1pt_increase', ascending=False)
    for _, row in suggestions_df.head(3).iterrows():
        if row['gain_per_1pt_increase'] > 0:
            print(f"  {row['aspect']}: +{row['gain_per_1pt_increase']:.3f} rating points per 1‑point improvement")
        else:
            print(f"  {row['aspect']}: already at maximum or no gain expected")

    return pred, suggestions_df

def interactive_new_hotel(aspects):
    print("== Plan Your New Hotel ==\n")
    print("For each aspect, rate your planned quality from 1 (poor) to 10 (excellent).")
    user_ratings = {}
    for aspect in aspects:
        while True:
            try:
                val = int(input(f"{aspect} (1-10): "))
                if 1 <= val <= 10:
                    user_ratings[aspect] = val
                    break
                else:
                    print("Please enter 1-10")
            except ValueError:
                print("Invalid input")
    return user_ratings

In [20]:
def hotel_investment_advisor(hotel_df, model_asym, pos_coef, neg_coef, aspects):
    while True:
        # Print menu at the start of each iteration
        print("\n" + "="*60)
        print("HOTEL INVESTMENT DECISION SUPPORT SYSTEM")
        print("="*60)
        print("1. Analyze an existing hotel")
        print("2. Score a new hotel (planned)")
        print("3. Exit")
        choice = input("\nSelect option (1/2/3): ")

        if choice == '1':
            hotel_list = hotel_df['hotel'].tolist()
            print("\nAvailable hotels (first 10):")
            for i, h in enumerate(hotel_list[:10]):
                print(f"{i+1}. {h}")
            if len(hotel_list) > 10:
                print("...")
            sel = input("Enter hotel name (or part of it): ")
            matches = [h for h in hotel_list if sel.lower() in h.lower()]
            if len(matches) == 0:
                print("No match found.")
            else:
                diagnose_hotel(matches[0], hotel_df, model_asym, pos_coef, neg_coef, aspects)

        elif choice == '2':
            user_ratings = interactive_new_hotel(aspects)
            score_new_hotel(user_ratings, model_asym, pos_coef, neg_coef, aspects)

        elif choice == '3':
            break
        else:
            print("Invalid choice.")

In [21]:
#Launch the Advisor
hotel_investment_advisor(hotel_df, model_asym, pos_coef, neg_coef, aspects)


HOTEL INVESTMENT DECISION SUPPORT SYSTEM
1. Analyze an existing hotel
2. Score a new hotel (planned)
3. Exit

Select option (1/2/3): 1

Available hotels (first 10):
1. 50 Albert Pl, Dehiwala-Mount Lavinia 10350
2. 519@jamtree 
3. 55 Lionel Edirisinghe Mawatha, Colombo 00500
4. AHAS GAWWA 
5. Aathma Colombo House
6. Ama HomeStay Guest Colombo
7. Amari Colombo, Sri Lanka
8. Anugaa in The City 
9. Artisan Villa
10. Atrium Leisure
...
Enter hotel name (or part of it): aathma

== Diagnosis for Aathma Colombo House ==
Current average rating: 9.73 / 10
Predicted rating: 9.55 / 10

**Strengths (positive impact):**
  facilities: +3.914 (pos rate 98.1%, neg rate 5.0%)
  staff: +0.883 (pos rate 95.0%, neg rate 1.3%)
  value for money: +0.742 (pos rate 96.2%, neg rate 3.1%)

**Weaknesses (negative impact):**
  check-in/out: -4.115 (pos rate 97.5%, neg rate 5.0%)
  cleanliness: -0.650 (pos rate 64.8%, neg rate 0.0%)
  location: -0.157 (pos rate 97.5%, neg rate 5.0%)

**Top investment priorities:**